# GRPO Training — Submission PGABL 

| Item | Detail |
|---|---|
| **Model Base** | `anggapradana/qwen2-5-1-5b-alpaca-id` (hasil fine-tuning) |
| **Dataset** | `Ichsan2895/alpaca-gpt4-indonesian` |
| **Metode** | GRPO (Group Relative Policy Optimization) |
| **Platform** | Kaggle (GPU T4) |

## 1. Instalasi Package

In [5]:
%%capture
!pip install unsloth
!pip install -U trl peft accelerate bitsandbytes datasets
!pip install rouge_score langdetect

## 2. Autentikasi HuggingFace

In [6]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token)
print("Login HuggingFace berhasil.")

Login HuggingFace berhasil.


## 3. Load Model Hasil Fine-tuning

Model dimuat dari HuggingFace (`anggapradana/qwen2-5-1-5b-alpaca-id`) lalu ditambahkan LoRA adapter baru untuk proses GRPO.
Konfigurasi QLoRA 4-bit dipertahankan untuk efisiensi VRAM.

In [7]:
from unsloth import FastLanguageModel
import torch

FINETUNED_MODEL = "anggapradana/qwen2-5-1-5b-alpaca-id"
MAX_SEQ_LENGTH  = 512   # Lebih pendek dari SFT untuk hemat VRAM saat GRPO

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FINETUNED_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print(f"Model '{FINETUNED_MODEL}' berhasil dimuat.")

==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model 'anggapradana/qwen2-5-1-5b-alpaca-id' berhasil dimuat.


In [8]:
# LoRA adapter baru untuk GRPO (r lebih kecil untuk hemat VRAM)
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=[
        # Multi-Head Attention
        "q_proj", "k_proj", "v_proj", "o_proj",
        # Feed Forward Network
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

Unsloth 2026.6.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


## 4. Persiapan Dataset untuk GRPO

Dataset yang sama (`Ichsan2895/alpaca-gpt4-indonesian`) digunakan sesuai ketentuan.
Prompt diformat dengan instruksi agar model berpikir menggunakan tag `<think>...</think>` sebelum menjawab.
Kolom `answer` disimpan sebagai referensi untuk reward correctness.

In [9]:
from datasets import load_dataset

raw_dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")
print(f"Total data: {len(raw_dataset)}")

GRPO_SYSTEM_PROMPT = (
    "Anda adalah asisten AI yang helpful dan jujur. "
    "Sebelum memberikan jawaban, tuliskan proses berpikir Anda secara detail "
    "dalam tag <think>...</think>. "
    "Kemudian berikan jawaban final yang jelas dalam Bahasa Indonesia."
)

def format_grpo_prompt(sample):
    messages = [
        {"role": "system", "content": GRPO_SYSTEM_PROMPT},
        {"role": "user",   "content": str(sample["input"]).strip()},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return {
        "prompt": prompt,
        "answer": str(sample["output"]).strip(),
    }

# Gunakan subset untuk efisiensi (GRPO lebih berat dari SFT)
dataset_grpo = raw_dataset.select(range(10000)).map(
    format_grpo_prompt,
    remove_columns=raw_dataset.column_names,
    num_proc=1,
)

print(f"Dataset GRPO siap: {len(dataset_grpo)} sampel")

README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Total data: 49969


Map (num_proc=1):   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset GRPO siap: 10000 sampel


In [10]:
# Tampilkan contoh prompt GRPO
print("=== Contoh Prompt GRPO ===")
print(dataset_grpo[0]["prompt"])
print("\n=== Ground Truth (answer) ===")
print(dataset_grpo[0]["answer"])

=== Contoh Prompt GRPO ===
<|im_start|>system
Anda adalah asisten AI yang helpful dan jujur. Sebelum memberikan jawaban, tuliskan proses berpikir Anda secara detail dalam tag <think>...</think>. Kemudian berikan jawaban final yang jelas dalam Bahasa Indonesia.<|im_end|>
<|im_start|>user
Saranlah slogan untuk kampanye daur ulang<|im_end|>
<|im_start|>assistant


=== Ground Truth (answer) ===
1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."
2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."
3. "Ubah sampahmu menjadi harta karun - Daur ulang!"
4. "Daur ulang untuk siklus kehidupan."
5. "Simpan sumber daya, daur ulang lebih banyak."


## 5. Reward Functions — Advanced Kriteria 1

Empat reward function sesuai ketentuan:
1. `format_reward_func` — reward bertahap untuk format `<think>...</think>`
2. `reasoning_length_reward` — reward berdasarkan panjang isi `<think>`
3. `correctness_reward` — reward berdasarkan kemiripan ROUGE-L dengan ground truth
4. `language_reward_func` — reward/penalti berdasarkan bahasa output

In [11]:
import re

def format_reward_func(completions, **kwargs):
    """
    Reward bertahap untuk format <think>...</think>:
    - +0.2 jika ada tag pembuka <think>
    - +0.3 jika ada tag penutup </think>
    - +1.0 jika format sempurna (di awal, ditutup benar, diikuti jawaban)
    - -0.5 penalti jika tag muncul lebih dari sekali (halusinasi)
    """
    rewards = []
    for completion in completions:
        reward = 0.0
        open_count  = completion.count("<think>")
        close_count = completion.count("</think>")

        # Penalti jika tag muncul lebih dari sekali
        if open_count > 1 or close_count > 1:
            reward -= 0.5
        else:
            if open_count == 1:
                reward += 0.2
            if close_count == 1:
                reward += 0.3

            # +1.0 jika format sempurna: dimulai <think>, ditutup </think>, diikuti jawaban
            perfect_pattern = r"^<think>.+?</think>\s*.+"
            if re.match(perfect_pattern, completion.strip(), re.DOTALL):
                reward = 1.0

        rewards.append(reward)
    return rewards


# Uji cepat
test_cases = [
    "<think>Ini proses berpikir saya.</think> Ini jawabannya.",   # perfect
    "<think>Berpikir...",                                          # hanya open
    "<think>A</think><think>B</think> jawaban",                   # halusinasi
    "Langsung jawab tanpa think.",                                 # tidak ada
]
print("Uji format_reward_func:")
for tc, r in zip(test_cases, format_reward_func(test_cases)):
    print(f"  {r:+.1f} | {tc[:50]}..." if len(tc) > 50 else f"  {r:+.1f} | {tc}")

Uji format_reward_func:
  +1.0 | <think>Ini proses berpikir saya.</think> Ini jawab...
  +0.2 | <think>Berpikir...
  -0.5 | <think>A</think><think>B</think> jawaban
  +0.0 | Langsung jawab tanpa think.


In [12]:
def reasoning_length_reward(completions, **kwargs):
    """
    Reward proporsional berdasarkan panjang isi <think>...</think>:
    - Tidak ada <think> atau isinya kosong : +0.0
    - Panjang < 50 karakter               : +0.2
    - Panjang 50–199 karakter             : +0.5
    - Panjang >= 200 karakter             : +1.0
    Toleran jika </think> tidak muncul (terpotong batas token).
    """
    rewards = []
    for completion in completions:
        # Coba ekstrak isi <think>
        # Toleran: jika </think> tidak ada, ambil semua setelah <think>
        if "<think>" not in completion:
            rewards.append(0.0)
            continue

        after_open = completion.split("<think>", 1)[1]
        if "</think>" in after_open:
            think_content = after_open.split("</think>", 1)[0]
        else:
            # Terpotong — ambil semua yang ada
            think_content = after_open

        think_content = think_content.strip()
        length = len(think_content)

        if length == 0:
            rewards.append(0.0)
        elif length < 50:
            rewards.append(0.2)
        elif length < 200:
            rewards.append(0.5)
        else:
            rewards.append(1.0)

    return rewards


# Uji cepat
test_cases_len = [
    "<think>Ok.</think> Jawaban.",
    "<think>" + "A" * 60 + "</think> Jawaban.",
    "<think>" + "B" * 250 + "</think> Jawaban.",
    "Tidak ada think.",
    "<think>" + "C" * 100,   # terpotong
]
print("Uji reasoning_length_reward:")
for tc, r in zip(test_cases_len, reasoning_length_reward(test_cases_len)):
    print(f"  {r:+.1f} | len think = {len(tc)}")

Uji reasoning_length_reward:
  +0.2 | len think = 27
  +0.5 | len think = 84
  +1.0 | len think = 274
  +0.0 | len think = 16
  +0.5 | len think = 107


In [13]:
from rouge_score import rouge_scorer as rs_module

_scorer = rs_module.RougeScorer(["rougeL"], use_stemmer=False)

def correctness_reward(completions, **kwargs):
    """
    Reward +1.0 jika output akhir (setelah </think>) mengandung ground truth
    atau memiliki kemiripan ROUGE-L >= 0.4.
    Jika di bawah threshold, diberikan skor ROUGE-L proporsional.
    """
    answers = kwargs.get("answer", [""] * len(completions))
    rewards = []
    for completion, answer in zip(completions, answers):
        # Ekstrak jawaban final (setelah </think>)
        if "</think>" in completion:
            final_answer = completion.split("</think>", 1)[-1].strip()
        else:
            final_answer = completion.strip()

        if not final_answer or not answer:
            rewards.append(0.0)
            continue

        # Cek apakah ground truth terkandung langsung
        if answer.lower()[:50] in final_answer.lower():
            rewards.append(1.0)
            continue

        # Gunakan ROUGE-L
        score = _scorer.score(answer, final_answer)["rougeL"].fmeasure
        rewards.append(1.0 if score >= 0.4 else round(score, 4))

    return rewards


print("correctness_reward didefinisikan.")

correctness_reward didefinisikan.


In [14]:
from langdetect import detect, LangDetectException

def language_reward_func(completions, **kwargs):
    """
    Reward berdasarkan bahasa output akhir (setelah </think>):
    - +1.0 jika murni Bahasa Indonesia
    - -0.5 jika terdeteksi Bahasa Inggris
    -  0.0 jika bahasa lain atau tidak terdeteksi
    """
    rewards = []
    for completion in completions:
        if "</think>" in completion:
            final_answer = completion.split("</think>", 1)[-1].strip()
        else:
            final_answer = completion.strip()

        if not final_answer:
            rewards.append(0.0)
            continue

        try:
            lang = detect(final_answer)
            if lang == "id":
                rewards.append(1.0)
            elif lang == "en":
                rewards.append(-0.5)
            else:
                rewards.append(0.0)
        except LangDetectException:
            rewards.append(0.0)

    return rewards


# Uji cepat
test_lang = [
    "<think>Saya berpikir.</think> Ini adalah jawaban dalam bahasa Indonesia.",
    "<think>I am thinking.</think> This is an answer in English.",
]
print("Uji language_reward_func:")
for tc, r in zip(test_lang, language_reward_func(test_lang)):
    print(f"  {r:+.1f} | {tc[:60]}")

Uji language_reward_func:
  +1.0 | <think>Saya berpikir.</think> Ini adalah jawaban dalam bahas
  -0.5 | <think>I am thinking.</think> This is an answer in English.


## 6. GRPOTrainer — Advanced Kriteria 1

Parameter kunci untuk mitigasi OOM:
- `num_generations=4` — jumlah completion yang digenerate per prompt
- `max_completion_length=256` — batas maksimal token per completion
- `per_device_train_batch_size=1`
- `gradient_accumulation_steps=4`

In [16]:
from trl import GRPOTrainer, GRPOConfig
from unsloth import is_bfloat16_supported

grpo_config = GRPOConfig(
    # Output
    output_dir="/kaggle/working/grpo_output",
    # Training
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=200,
    learning_rate=5e-6,
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_steps=100,
    seed=42,
    report_to="none",
    # GRPO-specific — mitigasi OOM
    num_generations=4,
    max_completion_length=256,
    temperature=0.7,
    beta=0.001,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,
        reasoning_length_reward,
        correctness_reward,
        language_reward_func,
    ],
    args=grpo_config,
    train_dataset=dataset_grpo,
)

print("GRPOTrainer siap.")

GRPOTrainer siap.


In [17]:
print("Memulai GRPO training...")
trainer_stats = trainer.train()

print("\nGRPO training selesai!")
print(f"Runtime: {trainer_stats.metrics['train_runtime']:.1f} detik")

Memulai GRPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 9,232,384 of 1,562,179,072 (0.59% trained)
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The atte

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
10,0.000000,1.658145,0.806523,180.825000,137.100000,207.600000,0.400000,119.183334,85.900000,144.300000,0.000017,0.280000,0.424960,0.420000,0.376942,0.183145,0.095388,0.775000,0.150000
20,0.000000,1.580183,0.606905,195.175000,141.700000,243.800000,0.425000,144.175002,116.100000,183.200000,0.000019,0.077500,0.357417,0.355000,0.378216,0.260182,0.131100,0.887500,0.225000
30,-0.000001,1.338248,0.427145,192.275000,132.600000,240.100000,0.350000,145.625002,107.000000,187.100000,0.000027,0.107500,0.158362,0.237500,0.347871,0.155747,0.061925,0.837500,0.240470
40,0.000000,1.715015,0.788229,147.650000,81.100000,195.400000,0.200000,130.691667,81.100000,163.700000,0.000043,0.222500,0.356879,0.287500,0.311603,0.367515,0.095385,0.837500,0.261603
50,0.000000,1.882812,0.829916,161.075000,108.700000,207.200000,0.300000,140.316667,108.700000,169.000000,0.000090,0.407500,0.508037,0.482500,0.472582,0.142812,0.109914,0.850000,0.300000
60,-0.000000,1.673492,0.546548,172.475000,107.600000,230.500000,0.350000,106.625000,56.400000,161.400000,0.000208,0.157500,0.320590,0.412500,0.329521,0.203493,0.102552,0.900000,0.136603
70,0.000001,1.924620,0.640294,170.725000,122.100000,217.200000,0.250000,128.225002,96.500000,158.900000,0.000767,0.282500,0.392247,0.517500,0.292243,0.224620,0.131949,0.900000,0.200000
80,0.000000,1.681452,0.543766,195.675000,137.400000,229.800000,0.500000,141.316667,111.800000,168.500000,0.000330,0.240000,0.366115,0.430000,0.335951,0.186453,0.108288,0.825000,0.265470
90,0.000001,1.961565,0.894364,169.575000,110.900000,228.800000,0.375000,104.041667,59.700000,156.700000,0.000989,0.320000,0.507472,0.492500,0.406765,0.261565,0.090068,0.887500,0.182735
100,0.000001,1.788743,0.762495,189.450000,142.300000,222.900000,0.450000,137.950002,116.700000,160.900000,0.001132,0.267500,0.375152,0.467500,0.377898,0.191243,0.112170,0.862500,0.232735


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


GRPO training selesai!
Runtime: 4001.9 detik


## 7. Push Model GRPO ke HuggingFace

In [18]:
HF_USERNAME   = "anggapradana"
GRPO_REPO     = "qwen2-5-1-5b-grpo-id"
FULL_GRPO_REPO = f"{HF_USERNAME}/{GRPO_REPO}"

print(f"Mendorong model GRPO ke: https://huggingface.co/{FULL_GRPO_REPO}")
print("Proses ini memakan waktu beberapa menit...")

model.push_to_hub_merged(
    FULL_GRPO_REPO,
    tokenizer,
    save_method="merged_16bit",
    token=hf_token,
)

print(f"\nModel GRPO berhasil di-push!")
print(f"Link: https://huggingface.co/{FULL_GRPO_REPO}")

Mendorong model GRPO ke: https://huggingface.co/anggapradana/qwen2-5-1-5b-grpo-id
Proses ini memakan waktu beberapa menit...


Unsloth: Restored added_tokens_decoder metadata in anggapradana/qwen2-5-1-5b-grpo-id/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:44<00:00, 44.62s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/anggapradana/qwen2-5-1-5b-grpo-id`

Model GRPO berhasil di-push!
Link: https://huggingface.co/anggapradana/qwen2-5-1-5b-grpo-id


In [19]:
# Update link_huggingface.txt dengan kedua link (fine-tuning + GRPO)
FINETUNED_REPO = "anggapradana/qwen2-5-1-5b-alpaca-id"

links = (
    f"https://huggingface.co/{FINETUNED_REPO}\n"
    f"https://huggingface.co/{FULL_GRPO_REPO}\n"
)

with open("/kaggle/working/link_huggingface.txt", "w") as f:
    f.write(links)

print("link_huggingface.txt diperbarui:")
print(links)

link_huggingface.txt diperbarui:
https://huggingface.co/anggapradana/qwen2-5-1-5b-alpaca-id
https://huggingface.co/anggapradana/qwen2-5-1-5b-grpo-id



## 8. Test Case Wajib — Advanced Kriteria 1

Model GRPO diuji dengan prompt wajib sesuai kriteria.
Output **harus** menampilkan proses reasoning dengan tag `<think>...</think>` sebelum jawaban final.

In [21]:
# Aktifkan mode inferensi
FastLanguageModel.for_inference(model)

TEST_PROMPT = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

messages = [
    {"role": "system", "content": GRPO_SYSTEM_PROMPT},
    {"role": "user",   "content": TEST_PROMPT},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

# Decode hanya bagian yang digenerate
generated = tokenizer.decode(
    output_ids[0][input_ids.shape[1]:],
    skip_special_tokens=True,
)

print("=" * 60)
print(f"PROMPT: {TEST_PROMPT}")
print("=" * 60)
print("OUTPUT MODEL GRPO:")
print(generated)
print("=" * 60)

# Verifikasi format (case-insensitive — model kadang menghasilkan <Think> kapital)
generated_lower = generated.lower()
has_think_open  = "<think>"  in generated_lower
has_think_close = "</think>" in generated_lower
print(f"\n✅ Tag <think> hadir   : {has_think_open}")
print(f"✅ Tag </think> hadir  : {has_think_close}")
print(f"✅ Format reasoning    : {has_think_open and has_think_close}")

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?
OUTPUT MODEL GRPO:
<Think>
Untuk memahami apakah Anda berhak mendapatkan gaji lembur atau tidak, pertama-tama perlu diketahui jenis pekerjaan apa yang dimiliki oleh manajer atau karyawan Anda. Jika pekerjaan tersebut diatur dengan aturan yang mengenakan penggantian waktu kerja, maka Anda mungkin akan mendapat uang lembur. Selain itu, jika pekerjaan tersebut melibatkan penawaran bepergian atau beban kebutuhan pribadi lainnya, ini juga bisa menjadi alasan bagi Anda untuk mendapatkan uang lembur.
Namun, jika pekerjaan tersebut tidak memiliki aturan penggantian waktu kerja atau penawaran bepergian, maka Anda tidak akan mendapatkan uang lembur. Namun, sebaiknya kontak dengan manajer Anda untuk memastikan bahwa Anda mendapatkan informasi yang akurat tentang status gaji lembur Anda.
</Think>

Jawaban:
Berdasarkan informasi yang diberikan, Anda mungkin memiliki hak mendapatkan uang lembur

## Ringkasan

| Kriteria | Status |
|---|---|
| Load model hasil fine-tuning dari HuggingFace | ✅ |
| GRPOTrainer dari TRL + Unsloth | ✅ |
| `format_reward_func` (reward bertahap +0.2/+0.3/+1.0, penalti -0.5) | ✅ |
| `reasoning_length_reward` (0.0/+0.2/+0.5/+1.0 berdasarkan panjang) | ✅ |
| `correctness_reward` (ROUGE-L vs ground truth) | ✅ |
| `language_reward_func` (+1.0 Indonesia, -0.5 Inggris) | ✅ |
| `num_generations` & `max_completion_length` untuk mitigasi OOM | ✅ |
| Push model GRPO ke HuggingFace | ✅ |
| Test case wajib prompt lembur + output `<think>` | ✅ |
| `link_huggingface.txt` berisi 2 link (fine-tuning + GRPO) | ✅ |